# Unit Tests — Ontario Sunshine List

Tests for each `src/` module. Each cell covers one function or behaviour.  
A cell prints `PASS <test name>` on success, or raises `AssertionError` with a message on failure.  
Run all cells top to bottom — if any cell fails, the error shows exactly which assertion broke.

In [ ]:
import sys
sys.path.insert(0, ".")

import numpy as np
import pandas as pd

def ok(name):
    print(f"  PASS  {name}")

## 1. data_loader

In [ ]:
from src.data_loader import _clean_salary, _extract_first_name

# _clean_salary: strips $, commas; coerces bad values to NaN
s = _clean_salary(pd.Series(["$123,456.00", "$100,000", "abc", None]))
assert s.iloc[0] == 123456.0,        f"expected 123456.0, got {s.iloc[0]}"
assert s.iloc[1] == 100000.0,        f"expected 100000.0, got {s.iloc[1]}"
assert pd.isna(s.iloc[2]),           "expected NaN for non-numeric string"
assert pd.isna(s.iloc[3]),           "expected NaN for None"
ok("_clean_salary: handles $, commas, non-numeric, None")

# _extract_first_name: skips leading initials
assert _extract_first_name("Jennifer")      == "Jennifer",  "plain name"
assert _extract_first_name("L. Jennifer")   == "Jennifer",  "skips leading initial"
assert _extract_first_name("A. B. Carlos")  == "Carlos",    "skips two initials"
assert _extract_first_name("")              == "",           "empty string"
assert _extract_first_name(None)            == "",           "None input"
ok("_extract_first_name: skips initials, handles edge cases")

In [ ]:
from src.data_loader import load_all
from pathlib import Path

# Skip if data folder has no CSVs (e.g. fresh clone without data)
csvs = list((Path(".") / "data").glob("*.csv"))
if not csvs:
    print("  SKIP  load_all — no CSVs found in data/")
else:
    df = load_all()
    required_cols = {"salary", "year", "sector", "employer", "job_title",
                     "first_name", "last_name", "first_name_clean"}
    missing = required_cols - set(df.columns)
    assert not missing,        f"load_all() missing columns: {missing}"
    assert (df["salary"] > 0).all(),  "all salaries should be > 0 after filtering"
    assert df["year"].notna().all(),   "no null years after filtering"
    assert df["year"].min() >= 2016,   f"unexpected year: {df['year'].min()}"
    assert df["year"].max() <= 2025,   f"unexpected year: {df['year'].max()}"
    assert len(df) > 100_000,         f"expected >100K records, got {len(df):,}"
    ok(f"load_all: {len(df):,} records, years {df['year'].min()}–{df['year'].max()}, {len(required_cols)} required columns present")

## 2. gender_inference

In [ ]:
from src.gender_inference import GenderClassifier
from pathlib import Path

ssa_files = list((Path(".") / "data" / "gender" / "ssa").glob("yob*.txt"))
if not ssa_files:
    print("  SKIP  GenderClassifier — SSA data not downloaded yet")
else:
    clf = GenderClassifier()

    # Unambiguous names — any reasonable classifier must get these right
    clear_female = ["Jennifer", "Patricia", "Linda", "Barbara", "Susan"]
    clear_male   = ["Michael", "James", "Robert", "William", "David"]

    for name in clear_female:
        label, conf = clf.classify(name)
        assert label == "Female", f"{name}: expected Female, got {label}"
    ok(f"classify: {clear_female} → Female")

    for name in clear_male:
        label, conf = clf.classify(name)
        assert label == "Male", f"{name}: expected Male, got {label}"
    ok(f"classify: {clear_male} → Male")

    # Confidence should be in [0.5, 1.0] for confirmed labels
    label, conf = clf.classify("Jennifer")
    assert conf is not None and 0.5 <= conf <= 1.0, f"confidence out of range: {conf}"
    ok("confidence is in [0.5, 1.0] for confirmed labels")

In [ ]:
    # classify_series: output shape and column contract
    names = pd.Series(["Jennifer", "Michael", "Zxqvbn", None, ""])
    result = clf.classify_series(names)

    assert list(result.columns) == ["gender", "gender_confidence", "gender_p_female"], \
        f"unexpected columns: {result.columns.tolist()}"
    assert len(result) == len(names), "output length must match input length"
    assert result.loc[0, "gender"] == "Female",    "Jennifer should be Female"
    assert result.loc[1, "gender"] == "Male",      "Michael should be Male"
    assert result.loc[2, "gender"] == "Uncertain", "gibberish should be Uncertain"
    assert result["gender"].isin(["Female", "Male", "Uncertain"]).all(), \
        "all labels must be Female / Male / Uncertain"
    ok("classify_series: correct columns, length, and label values")

## 3. clustering

In [ ]:
from src.clustering import normalize_titles

titles = pd.Series(["Senior  Engineer", "NURSE PRACTITIONER", "  Director  ", ""])
result = normalize_titles(titles)

assert result.iloc[0] == "senior engineer",      f"got: {result.iloc[0]}"
assert result.iloc[1] == "nurse practitioner",   f"got: {result.iloc[1]}"
assert result.iloc[2] == "director",             f"got: {result.iloc[2]}"
assert result.iloc[3] == "",                     f"got: {result.iloc[3]}"
ok("normalize_titles: lowercase, strip, collapse whitespace")

In [ ]:
from src.clustering import get_embeddings
from pathlib import Path
import tempfile

titles = ["registered nurse", "software engineer", "finance director"]

with tempfile.TemporaryDirectory() as tmp:
    cache = Path(tmp) / "test_emb.pkl"
    embs = get_embeddings(titles, cache_path=cache)

    # Shape: one row per title, 384 dims for all-MiniLM-L6-v2
    assert embs.shape == (3, 384), f"unexpected shape: {embs.shape}"

    # L2-normalized: each row norm ≈ 1.0
    norms = np.linalg.norm(embs, axis=1)
    assert np.allclose(norms, 1.0, atol=1e-5), f"embeddings not L2-normalized: {norms}"

    # Cache round-trip: second call loads from disk
    embs2 = get_embeddings(titles, cache_path=cache)
    assert np.array_equal(embs, embs2), "cache round-trip returned different embeddings"

ok("get_embeddings: correct shape, L2-normalized, cache round-trip")

In [ ]:
from src.clustering import cluster_kmeans, cluster_hdbscan, cluster_summary

# Build a tiny synthetic df with two obvious clusters
rng = np.random.default_rng(42)
embs_a = rng.normal([1, 0], 0.05, size=(20, 2))  # cluster A
embs_b = rng.normal([0, 1], 0.05, size=(20, 2))  # cluster B
embs_test = np.vstack([embs_a, embs_b])

labels = cluster_kmeans(embs_test, k=2)
assert len(labels) == 40,                         "one label per embedding"
assert set(labels) == {0, 1},                     f"expected labels {{0,1}}, got {set(labels)}"
# Both clusters should be non-trivial (at least 5 points each)
assert min(np.bincount(labels)) >= 5,             f"a cluster has <5 points: {np.bincount(labels)}"
ok("cluster_kmeans: correct label count and non-trivial clusters on separable data")

hdb_labels = cluster_hdbscan(embs_test, min_cluster_size=5)
assert len(hdb_labels) == 40,                     "one label per embedding"
n_clusters = len(set(hdb_labels) - {-1})
assert n_clusters >= 1,                           f"HDBSCAN found no clusters: {set(hdb_labels)}"
ok(f"cluster_hdbscan: found {n_clusters} cluster(s) on separable data")

In [ ]:
synthetic_df = pd.DataFrame({
    "job_title_norm": ["nurse"] * 20 + ["engineer"] * 20,
    "cluster_km":     [0] * 20 + [1] * 20,
    "salary":         rng.normal(110_000, 10_000, 40),
    "gender":         (["Female"] * 10 + ["Male"] * 10) * 2,
})

summary = cluster_summary(synthetic_df, cluster_col="cluster_km", top_n=3)

assert len(summary) == 2,                              "one row per cluster"
assert "top_titles" in summary.columns,                "missing top_titles"
assert "median_salary" in summary.columns,             "missing median_salary"
assert "pct_female" in summary.columns,                "missing pct_female"
assert summary["n_records"].sum() == 40,               "record count mismatch"
assert (summary["pct_female"].dropna().between(0, 1)).all(), "pct_female out of [0,1]"
ok("cluster_summary: correct row count, columns, and value ranges")

## 4. stats

In [ ]:
from src.stats import raw_gap_by_cluster, apply_bh_fdr

rng = np.random.default_rng(0)

# Cluster 0: clear male advantage (males earn ~20K more)
# Cluster 1: no gap
# Cluster 2: sparse (only 3 females)
stats_df = pd.DataFrame({
    "cluster_km": [0]*40 + [1]*40 + [2]*13,
    "gender":     (["Female"]*20 + ["Male"]*20) +
                  (["Female"]*20 + ["Male"]*20) +
                  (["Female"]*3  + ["Male"]*10),
    "salary":     np.concatenate([
        rng.normal(110_000, 5_000, 20),   # cluster 0 female
        rng.normal(130_000, 5_000, 20),   # cluster 0 male
        rng.normal(120_000, 5_000, 20),   # cluster 1 female
        rng.normal(120_000, 5_000, 20),   # cluster 1 male
        rng.normal(115_000, 5_000, 13),   # cluster 2 (sparse)
    ]),
    "sector": "Health",
    "year": 2022,
})

gap = raw_gap_by_cluster(stats_df, cluster_col="cluster_km")

assert len(gap) == 3,                              "one row per cluster"
assert set(gap.columns) >= {"cluster","n_female","n_male","gap_pct","cles","sparse","mw_pvalue"}, \
    f"missing columns: {set(gap.columns)}"

c0 = gap[gap["cluster"] == 0].iloc[0]
c2 = gap[gap["cluster"] == 2].iloc[0]
assert c0["sparse"] == False,                      "cluster 0 should not be sparse"
assert c2["sparse"] == True,                       "cluster 2 should be sparse (3 females)"
assert c0["gap_pct"] < 0,                          f"cluster 0: expected male-favoring gap, got {c0['gap_pct']:.3f}"
assert c0["cles"] < 0.5,                           f"cluster 0: CLES should be <0.5, got {c0['cles']:.3f}"
assert pd.isna(c2["mw_pvalue"]),                   "sparse cluster should have NaN p-value"
ok("raw_gap_by_cluster: gap direction, sparse flag, NaN p-value for sparse clusters")

In [ ]:
gap_fdr = apply_bh_fdr(gap)

assert "bh_pvalue"      in gap_fdr.columns, "missing bh_pvalue column"
assert "bh_significant" in gap_fdr.columns, "missing bh_significant column"

# Sparse cluster should still have NaN after FDR
c2_fdr = gap_fdr[gap_fdr["cluster"] == 2].iloc[0]
assert pd.isna(c2_fdr["bh_pvalue"]),        "sparse cluster bh_pvalue should be NaN"
assert c2_fdr["bh_significant"] == False,   "sparse cluster should not be significant"

# Cluster 0 has a huge gap — should survive FDR correction
c0_fdr = gap_fdr[gap_fdr["cluster"] == 0].iloc[0]
assert c0_fdr["bh_significant"] == True,    "cluster 0 large gap should be significant after FDR"
ok("apply_bh_fdr: adds corrected p-values, sparse clusters untouched, obvious gap survives FDR")

In [ ]:
from src.stats import regression_adjusted_gap

reg = regression_adjusted_gap(stats_df, cluster_col="cluster_km")

assert reg.nobs > 0,                          "regression fit on zero observations"
assert "gender_fac[T.Male]" in reg.params,    "gender coefficient missing from model"
coef = reg.params["gender_fac[T.Male]"]
assert coef > 0,                              f"expected positive male premium, got {coef:.4f}"
assert 0.0 < reg.rsquared <= 1.0,             f"R² out of range: {reg.rsquared:.4f}"
ok(f"regression_adjusted_gap: male premium = {coef*100:+.1f}%, R² = {reg.rsquared:.3f}")

## Summary

If all cells above printed `PASS`, every module is functioning correctly.  
Any `AssertionError` shows exactly which check failed and what value was actually returned — fix the issue in `src/`, restart the kernel, and re-run from the top.